# Alignment-Aware Hilbert Residual Screen

This notebook promotes the first above-gate alignment result into the Huber + additive Hilbert residual path. It uses the log-plus-trajectory nearest analog from the refinement screen and tests whether analog-aware smooth residual heads improve beyond a direct gated analog correction.

## 1. Setup

In [1]:
"Configure imports, paths, and alignment-aware residual candidates."

from pathlib import Path
import json
import shutil
import time

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import HuberRegressor, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, RobustScaler, StandardScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "context").exists():
    PROJECT_ROOT = next(parent for parent in PROJECT_ROOT.parents if (parent / "context").exists())

DATE_STAMP = "2026-06-23"
RUN_ID = f"{DATE_STAMP}__alignment_hsgp_residual"
ARTIFACT_TABLES = PROJECT_ROOT / "artifacts" / "tables"
ARTIFACT_REPORTS = PROJECT_ROOT / "artifacts" / "reports"
ARTIFACT_LOGS = PROJECT_ROOT / "artifacts" / "logs" / "runs" / RUN_ID
for path in [ARTIFACT_TABLES, ARTIFACT_REPORTS, ARTIFACT_LOGS]:
    path.mkdir(parents=True, exist_ok=True)

RESIDUAL_PATH = ARTIFACT_TABLES / "model__hsgp_residual_on_gam_huber_residual_sample__2026-06-22.csv"
ANALOG_PATH = ARTIFACT_TABLES / "model__analog_selection_refinement_residual_sample__2026-06-23.csv"
TARGET_CANDIDATE = "huber_plus_hsgp_dps_m8"
ANALOG_CANDIDATE = "log_plus_trajectory_k1_progress_gate"
LOCAL_REFERENCE_RMSE = 14.3586
PUBLIC_REFERENCE_RMSE = 14.514
OFFSET_CLIP = 14.0
RANDOM_SEED = 20260623

CANDIDATES = [
    {"name": "baseline_huber_hilbert", "kind": "baseline", "alpha": 0.0, "shrink": 0.0},
    {"name": "direct_analog_gate", "kind": "direct_analog", "alpha": 0.0, "shrink": 1.0},
    {"name": "analog_row_ridge_a300_s050", "kind": "analog_row_ridge", "alpha": 300.0, "shrink": 0.50},
    {"name": "analog_row_ridge_a1000_s050", "kind": "analog_row_ridge", "alpha": 1000.0, "shrink": 0.50},
    {"name": "analog_row_huber_a010_s050", "kind": "analog_row_huber", "alpha": 0.10, "shrink": 0.50},
    {"name": "analog_poly2_ridge_a1500_s035", "kind": "analog_poly2_ridge", "alpha": 1500.0, "shrink": 0.35},
    {"name": "analog_rbf_m64_g025_a300_s035", "kind": "analog_rbf_ridge", "alpha": 300.0, "gamma": 0.25, "components": 64, "shrink": 0.35},
    {"name": "analog_rbf_m96_g050_a500_s025", "kind": "analog_rbf_ridge", "alpha": 500.0, "gamma": 0.50, "components": 96, "shrink": 0.25},
    {"name": "analog_rbf_m128_g075_a800_s025", "kind": "analog_rbf_ridge", "alpha": 800.0, "gamma": 0.75, "components": 128, "shrink": 0.25},
]

print(f"project_root={PROJECT_ROOT}")
print(f"run_id={RUN_ID}")
print(f"candidate_count={len(CANDIDATES)}")

project_root=C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling
run_id=2026-06-23__alignment_hsgp_residual
candidate_count=9


## 2. Load Current Residuals And Refined Analog Signal

In [2]:
"Load the current Huber + Hilbert residual sample and merge refined analog corrections."

residuals_all = pd.read_csv(RESIDUAL_PATH)
base = residuals_all.loc[residuals_all["candidate"].eq(TARGET_CANDIDATE)].copy()
for col in ["fold", "row_idx", "distance_from_boundary_md", "eval_progress", "abs_projected_delta_10", "y_true", "y_pred", "residual"]:
    base[col] = pd.to_numeric(base[col], errors="coerce")
base["fold"] = base["fold"].astype(int)
base["row_idx"] = base["row_idx"].astype(int)
base["distance_log1p"] = np.log1p(base["distance_from_boundary_md"].clip(lower=0))
base["baseline_pred"] = base["y_pred"]
base["baseline_residual"] = base["y_true"] - base["baseline_pred"]
base["pseudo_zone"] = pd.cut(base["eval_progress"], bins=[-0.001, 0.25, 0.50, 0.75, 1.001], labels=["z0", "z1", "z2", "z3"])

analog_all = pd.read_csv(ANALOG_PATH)
analog = analog_all.loc[analog_all["candidate"].eq(ANALOG_CANDIDATE), ["fold", "wellname", "row_idx", "selected_analogs", "analog_distance", "analog_gap", "analog_shrink", "analog_correction"]].copy()
for col in ["fold", "row_idx", "analog_distance", "analog_gap", "analog_shrink", "analog_correction"]:
    analog[col] = pd.to_numeric(analog[col], errors="coerce")
analog["fold"] = analog["fold"].astype(int)
analog["row_idx"] = analog["row_idx"].astype(int)

model_frame = base.merge(analog, on=["fold", "wellname", "row_idx"], how="left")
model_frame["analog_gap_clipped"] = model_frame["analog_gap"].clip(-140.0, 140.0)
model_frame["analog_gap_abs"] = model_frame["analog_gap_clipped"].abs()
model_frame["analog_correction"] = model_frame["analog_correction"].fillna(0.0)
model_frame["analog_distance"] = model_frame["analog_distance"].fillna(model_frame["analog_distance"].median())

print(f"rows={len(model_frame):,}")
print(f"wells={model_frame['wellname'].nunique():,}")
print(model_frame[["baseline_residual", "analog_gap_clipped", "analog_correction"]].corr().round(4).to_string())

rows=5,400
wells=773
                    baseline_residual  analog_gap_clipped  analog_correction
baseline_residual              1.0000              0.1177             0.1255
analog_gap_clipped             0.1177              1.0000             0.9060
analog_correction              0.1255              0.9060             1.0000


## 3. Feature Matrix

In [3]:
"Build row, analog, zone, and smooth residual features."

ZONE_COLUMNS = ["zone_z0", "zone_z1", "zone_z2", "zone_z3"]
BASE_FEATURES = ["eval_progress", "distance_log1p", "abs_projected_delta_10"]
ANALOG_FEATURES = ["analog_gap_clipped", "analog_gap_abs", "analog_shrink", "analog_correction", "analog_distance"]


def add_features(frame):
    out = frame.copy()
    zones = pd.get_dummies(out["pseudo_zone"].astype(str), prefix="zone")
    for col in ZONE_COLUMNS:
        out[col] = zones[col] if col in zones.columns else 0.0
    out["analog_gap__x__progress"] = out["analog_gap_clipped"] * out["eval_progress"]
    out["analog_gap__x__distance_log1p"] = out["analog_gap_clipped"] * out["distance_log1p"]
    out["analog_gap__x__slope_instability"] = out["analog_gap_clipped"] * out["abs_projected_delta_10"]
    out["analog_correction__x__progress"] = out["analog_correction"] * out["eval_progress"]
    for zone_col in ZONE_COLUMNS:
        out[f"analog_correction__x__{zone_col}"] = out["analog_correction"] * out[zone_col]
    return out

FEATURE_COLUMNS = BASE_FEATURES + ANALOG_FEATURES + ZONE_COLUMNS + [
    "analog_gap__x__progress",
    "analog_gap__x__distance_log1p",
    "analog_gap__x__slope_instability",
    "analog_correction__x__progress",
] + [f"analog_correction__x__{zone_col}" for zone_col in ZONE_COLUMNS]
RBF_COLUMNS = ["eval_progress", "distance_log1p", "abs_projected_delta_10", "analog_gap_clipped", "analog_correction"]

model_features = add_features(model_frame)
print(pd.DataFrame({"feature": FEATURE_COLUMNS}).to_string(index=False))

                         feature
                   eval_progress
                  distance_log1p
          abs_projected_delta_10
              analog_gap_clipped
                  analog_gap_abs
                   analog_shrink
               analog_correction
                 analog_distance
                         zone_z0
                         zone_z1
                         zone_z2
                         zone_z3
         analog_gap__x__progress
   analog_gap__x__distance_log1p
analog_gap__x__slope_instability
  analog_correction__x__progress
   analog_correction__x__zone_z0
   analog_correction__x__zone_z1
   analog_correction__x__zone_z2
   analog_correction__x__zone_z3


## 4. Out-Of-Fold Residual Evaluation

In [4]:
"Train analog-aware residual heads on other folds and score held-out folds."

start_time = time.time()
metric_rows = []
prediction_rows = []
for candidate in CANDIDATES:
    for fold in sorted(model_features["fold"].unique()):
        train_rows = model_features.loc[~model_features["fold"].eq(fold)].copy()
        valid_rows = model_features.loc[model_features["fold"].eq(fold)].copy()
        raw_correction = np.zeros(len(valid_rows), dtype=float)
        if candidate["kind"] == "direct_analog":
            raw_correction = valid_rows["analog_correction"].to_numpy(dtype=float)
        elif candidate["kind"] == "analog_row_ridge":
            model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", Ridge(alpha=candidate["alpha"]))])
            model.fit(train_rows[FEATURE_COLUMNS], train_rows["baseline_residual"])
            raw_correction = model.predict(valid_rows[FEATURE_COLUMNS])
        elif candidate["kind"] == "analog_row_huber":
            model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", RobustScaler()), ("model", HuberRegressor(alpha=candidate["alpha"], epsilon=1.35, max_iter=500))])
            model.fit(train_rows[FEATURE_COLUMNS], train_rows["baseline_residual"])
            raw_correction = model.predict(valid_rows[FEATURE_COLUMNS])
        elif candidate["kind"] == "analog_poly2_ridge":
            model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("poly", PolynomialFeatures(degree=2, include_bias=False)), ("model", Ridge(alpha=candidate["alpha"]))])
            model.fit(train_rows[FEATURE_COLUMNS], train_rows["baseline_residual"])
            raw_correction = model.predict(valid_rows[FEATURE_COLUMNS])
        elif candidate["kind"] == "analog_rbf_ridge":
            model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("rbf", RBFSampler(gamma=candidate["gamma"], n_components=candidate["components"], random_state=RANDOM_SEED + int(fold))), ("model", Ridge(alpha=candidate["alpha"]))])
            model.fit(train_rows[RBF_COLUMNS], train_rows["baseline_residual"])
            raw_correction = model.predict(valid_rows[RBF_COLUMNS])
        correction = np.clip(raw_correction * candidate.get("shrink", 1.0), -OFFSET_CLIP, OFFSET_CLIP)
        y_true = valid_rows["y_true"].to_numpy()
        y_pred = valid_rows["baseline_pred"].to_numpy() + correction
        metric_rows.append(
            {
                "candidate": candidate["name"],
                "kind": candidate["kind"],
                "fold": int(fold),
                "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
                "mae": float(mean_absolute_error(y_true, y_pred)),
                "n_rows": int(len(valid_rows)),
                "n_wells": int(valid_rows["wellname"].nunique()),
                "mean_correction": float(np.mean(correction)),
                "std_correction": float(np.std(correction)),
                "max_abs_correction": float(np.max(np.abs(correction))) if len(correction) else 0.0,
            }
        )
        out = valid_rows[["fold", "wellname", "row_idx", "pseudo_zone", "y_true", "baseline_pred", "baseline_residual", "eval_progress", "distance_log1p", "abs_projected_delta_10", "analog_gap_clipped", "analog_correction", "selected_analogs"]].copy()
        out["candidate"] = candidate["name"]
        out["alignment_residual_correction"] = correction
        out["y_pred"] = y_pred
        out["residual_after_alignment"] = out["y_true"] - out["y_pred"]
        prediction_rows.append(out)

fold_metrics = pd.DataFrame(metric_rows)
predictions = pd.concat(prediction_rows, ignore_index=True)
summary = (
    fold_metrics.groupby(["candidate", "kind"], as_index=False)
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        mae_mean=("mae", "mean"),
        mean_correction=("mean_correction", "mean"),
        std_correction=("std_correction", "mean"),
        max_abs_correction=("max_abs_correction", "max"),
        n_rows=("n_rows", "sum"),
        n_wells=("n_wells", "sum"),
    )
)
baseline_rmse = float(summary.loc[summary["candidate"].eq("baseline_huber_hilbert"), "rmse_mean"].iloc[0])
summary["rmse_delta_vs_sample_baseline"] = summary["rmse_mean"] - baseline_rmse
summary = summary.sort_values(["rmse_mean", "candidate"]).reset_index(drop=True)
train_time_s = time.time() - start_time
print(summary.to_string(index=False))

                     candidate               kind  rmse_mean  rmse_std  mae_mean  mean_correction  std_correction  max_abs_correction  n_rows  n_wells  rmse_delta_vs_sample_baseline
            direct_analog_gate      direct_analog  14.638289  0.411922 10.353734        -0.043459        1.207596            8.494656    5400      773                      -0.100256
    analog_row_ridge_a300_s050   analog_row_ridge  14.662878  0.437523 10.387546         0.118941        1.085605           14.000000    5400      773                      -0.075667
   analog_row_ridge_a1000_s050   analog_row_ridge  14.667192  0.439496 10.385563         0.120036        1.009437           14.000000    5400      773                      -0.071353
 analog_poly2_ridge_a1500_s035 analog_poly2_ridge  14.668348  0.439053 10.376764         0.080443        0.948466           14.000000    5400      773                      -0.070197
    analog_row_huber_a010_s050   analog_row_huber  14.684683  0.436384 10.404055         0

## 5. Diagnostics And Export

In [5]:
"Summarize alignment-aware residual diagnostics and export artifacts."

best_candidate = summary.iloc[0]["candidate"]
best_row = summary.iloc[0]
baseline_row = summary.loc[summary["candidate"].eq("baseline_huber_hilbert")].iloc[0]
best_predictions = predictions.loc[predictions["candidate"].eq(best_candidate)].copy()
zone_summary = (
    best_predictions.groupby("pseudo_zone", observed=False)
    .agg(
        rmse_before=("baseline_residual", lambda s: float(np.sqrt(np.mean(np.asarray(s) ** 2)))),
        rmse_after=("residual_after_alignment", lambda s: float(np.sqrt(np.mean(np.asarray(s) ** 2)))),
        mean_correction=("alignment_residual_correction", "mean"),
        rows=("row_idx", "size"),
    )
    .reset_index()
)
zone_summary["rmse_delta"] = zone_summary["rmse_after"] - zone_summary["rmse_before"]

well_before = model_features.groupby(["fold", "wellname"], as_index=False).agg(mean_residual_before=("baseline_residual", "mean"))
well_after = best_predictions.groupby(["fold", "wellname"], as_index=False).agg(mean_residual_after=("residual_after_alignment", "mean"), mean_alignment_correction=("alignment_residual_correction", "mean"))
well_diagnostics = well_after.merge(well_before, on=["fold", "wellname"], how="left")
well_diagnostics["abs_bias_delta"] = well_diagnostics["mean_residual_after"].abs() - well_diagnostics["mean_residual_before"].abs()

findings = pd.DataFrame(
    [
        {"finding": "best_candidate", "value": f"{best_candidate} RMSE {best_row['rmse_mean']:.4f} vs sample baseline {baseline_row['rmse_mean']:.4f}", "modeling_implication": "Promote to Kaggle-safe packaging only if alignment-aware residuals beat the direct analog correction and stay above the gate."},
        {"finding": "sample_baseline_delta", "value": f"best candidate delta {best_row['rmse_delta_vs_sample_baseline']:.4f}", "modeling_implication": "This measures whether analog selection remains useful after the Huber + Hilbert residual model."},
        {"finding": "best_kind", "value": str(best_row["kind"]), "modeling_implication": "Identifies whether direct analog transfer or learned smooth residual heads carry the signal."},
        {"finding": "well_bias_change", "value": f"median absolute well-bias delta {well_diagnostics['abs_bias_delta'].median():.4f}", "modeling_implication": "Negative is good; positive means the alignment residual correction adds well-level noise."},
    ]
)

summary_path = ARTIFACT_TABLES / f"model__alignment_hsgp_residual_summary__{DATE_STAMP}.csv"
fold_path = ARTIFACT_TABLES / f"model__alignment_hsgp_residual_fold_metrics__{DATE_STAMP}.csv"
pred_path = ARTIFACT_TABLES / f"model__alignment_hsgp_residual_residual_sample__{DATE_STAMP}.csv"
well_path = ARTIFACT_TABLES / f"model__alignment_hsgp_residual_well_diagnostics__{DATE_STAMP}.csv"
zone_path = ARTIFACT_TABLES / f"model__alignment_hsgp_residual_zone_summary__{DATE_STAMP}.csv"
findings_path = ARTIFACT_TABLES / f"model__alignment_hsgp_residual_findings__{DATE_STAMP}.csv"
report_path = ARTIFACT_REPORTS / f"model__alignment_hsgp_residual__{DATE_STAMP}.md"
comparison_path = ARTIFACT_TABLES / "model_comparison.csv"

summary.to_csv(summary_path, index=False)
fold_metrics.to_csv(fold_path, index=False)
predictions.to_csv(pred_path, index=False)
well_diagnostics.to_csv(well_path, index=False)
zone_summary.to_csv(zone_path, index=False)
findings.to_csv(findings_path, index=False)

report = f"""# Alignment-Aware Hilbert Residual Screen

Run ID: `{RUN_ID}`

Result

Best candidate: `{best_candidate}`

- Same-sample baseline RMSE: **{baseline_row['rmse_mean']:.4f}**
- Best RMSE: **{best_row['rmse_mean']:.4f}**
- Delta vs same-sample baseline: **{best_row['rmse_delta_vs_sample_baseline']:.4f}**
- Current full local reference: **{LOCAL_REFERENCE_RMSE:.4f}**
- Current public reference: **{PUBLIC_REFERENCE_RMSE:.3f}**

Interpretation

This screen tests whether the above-gate log-plus-trajectory analog selector still adds value after the current Huber + additive Hilbert residual model. It is a residual-sample diagnostic, not yet a self-contained Kaggle notebook.
"""
report_path.write_text(report, encoding="utf-8")

comparison_row = pd.DataFrame(
    [
        {
            "run_id": RUN_ID,
            "model_family": "alignment_hsgp_residual",
            "model_name": best_candidate,
            "cv_rmse_mean": float(best_row["rmse_mean"]),
            "cv_rmse_std": float(best_row["rmse_std"]),
            "train_time_s": float(train_time_s),
            "n_features": int(len(FEATURE_COLUMNS)),
            "notes": "Alignment-aware residual heads using log-plus-trajectory nearest analog signal on current Huber + additive Hilbert residual sample",
        }
    ]
)
comparison = pd.read_csv(comparison_path) if comparison_path.exists() else pd.DataFrame()
if len(comparison):
    comparison = comparison.loc[~comparison["run_id"].eq(RUN_ID)]
comparison = pd.concat([comparison, comparison_row], ignore_index=True)
comparison.to_csv(comparison_path, index=False)

for src in [summary_path, fold_path, pred_path, well_path, zone_path, findings_path, report_path, comparison_path]:
    shutil.copy2(src, ARTIFACT_LOGS / src.name)

config = {"run_id": RUN_ID, "target_candidate": TARGET_CANDIDATE, "analog_candidate": ANALOG_CANDIDATE, "offset_clip": OFFSET_CLIP, "candidates": CANDIDATES, "feature_columns": FEATURE_COLUMNS, "rbf_columns": RBF_COLUMNS, "local_reference_rmse": LOCAL_REFERENCE_RMSE, "public_reference_rmse": PUBLIC_REFERENCE_RMSE}
(ARTIFACT_LOGS / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

print(findings.to_string(index=False))
print(zone_summary.to_string(index=False))
print(f"wrote {summary_path}")
print(f"wrote run bundle {ARTIFACT_LOGS}")

              finding                                                      value                                                                                                          modeling_implication
       best_candidate direct_analog_gate RMSE 14.6383 vs sample baseline 14.7385 Promote to Kaggle-safe packaging only if alignment-aware residuals beat the direct analog correction and stay above the gate.
sample_baseline_delta                               best candidate delta -0.1003                               This measures whether analog selection remains useful after the Huber + Hilbert residual model.
            best_kind                                              direct_analog                                  Identifies whether direct analog transfer or learned smooth residual heads carry the signal.
     well_bias_change                    median absolute well-bias delta -0.0963                                     Negative is good; positive means the alignment residual

## 6. Conclusion

In [6]:
"Print final decision summary."

print(f"best_candidate={best_candidate}")
print(f"sample_baseline_rmse={baseline_row['rmse_mean']:.4f}")
print(f"best_rmse={best_row['rmse_mean']:.4f}")
print(f"delta_vs_sample_baseline={best_row['rmse_delta_vs_sample_baseline']:.4f}")
print(f"promotion_gate_cleared={best_row['rmse_delta_vs_sample_baseline'] <= -0.05}")

best_candidate=direct_analog_gate
sample_baseline_rmse=14.7385
best_rmse=14.6383
delta_vs_sample_baseline=-0.1003
promotion_gate_cleared=True
